# 面试问题：Softmax 与 Cross-Entropy 为什么要用 LogSumExp，梯度 `p-y` 怎样推导并实现？

**一句话回答**：`softmax(z)_i=exp(z_i)/Σexp(z_j)`，直接指数会溢出；减去每行最大值不改变概率。交叉熵最好直接从 logits 计算 `logsumexp(z)-z_target`，其梯度是 `softmax(z)-one_hot(y)`；batch、mask、class weight 和 label smoothing 会进一步改变缩放，不能先做错误 reduction 再补 mask。

下面用 NumPy 与 PyTorch 基础算子实现稳定前向、有限差分梯度、mask/权重和一个自定义 `nn.Module`。

In [ ]:
import hashlib, json, math
import numpy as np
import torch
from torch import nn

SEED94=9401; rng94=np.random.default_rng(SEED94); torch.manual_seed(SEED94)
assert SEED94==9401
assert np.exp(0)==1
assert torch.__version__

## 1. 稳定 Softmax

对每行减去 `m=max(z)`：分子分母都乘 `exp(-m)`，概率不变，但最大指数变成 1，避免正溢出。仍要检查 logits NaN/Inf；全为 `-inf` 的 masked 行没有合法分布，应由上游保证至少一个可选项。

Softmax 对整体平移不变，但温度缩放不是平移，会改变熵。

In [ ]:
def softmax94(logits):
    z=np.asarray(logits,float)
    if z.ndim<1 or z.shape[-1]<1 or not np.isfinite(z).all(): raise ValueError("logit_contract")
    shifted=z-np.max(z,axis=-1,keepdims=True); exp=np.exp(shifted); return exp/exp.sum(axis=-1,keepdims=True)
p94=softmax94([[1000,1001,999],[-1000,-1001,-999]])
assert np.allclose(p94.sum(1),1) and np.isfinite(p94).all()
assert np.allclose(softmax94([[1,2,3]]),softmax94([[101,102,103]]))
try: softmax94([[1,np.nan]]); raise AssertionError("nan accepted")
except ValueError as e: assert str(e)=="logit_contract"

## 2. LogSumExp 与单样本交叉熵

`logsumexp(z)=m+log Σexp(z-m)`。hard-label CE 为 `LSE(z)-z_y`，不需要显式构造概率再取 log，避免正确类概率下溢成 0。多分类 label 必须是 `[0,C)` 整数。

极大正确 logit 的 loss 接近 0；极大错误 logit 的 loss 很大但有限。

In [ ]:
def logsumexp94(z):
    z=np.asarray(z,float); m=np.max(z,axis=-1,keepdims=True); return np.squeeze(m+np.log(np.exp(z-m).sum(axis=-1,keepdims=True)),-1)
def cross_entropy94(logits,targets):
    z=np.asarray(logits,float); y=np.asarray(targets,int)
    if z.ndim!=2 or y.shape!=(len(z),) or np.any(y<0) or np.any(y>=z.shape[1]): raise ValueError("target_contract")
    return logsumexp94(z)-z[np.arange(len(z)),y]
losses94=cross_entropy94([[1000,999],[-1000,-999]],[0,1])
assert np.allclose(losses94,[math.log1p(math.exp(-1))]*2)
assert cross_entropy94([[1000,-1000]],[0])[0]<1e-10
assert np.isfinite(cross_entropy94([[-10000,10000]],[0]))

## 3. 梯度为何是 `p-y`

`∂LSE/∂z_i=p_i`，正确类项 `-z_y` 再贡献 `-1`，所以 `∂L/∂z=p-one_hot(y)`。所有类别梯度之和为 0，对应整体平移不改变 loss。预测越自信且错误，错误类正梯度和正确类负梯度越大。

用中心有限差分作为独立数值 oracle。

In [ ]:
def ce_grad94(logits,targets):
    p=softmax94(logits); g=p.copy(); g[np.arange(len(g)),np.asarray(targets,int)]-=1; return g
z94=rng94.normal(size=(2,4)); y94=np.array([1,3]); analytic94=ce_grad94(z94,y94); numeric94=np.zeros_like(z94); eps94=1e-5
for i in range(z94.shape[0]):
    for j in range(z94.shape[1]):
        plus=z94.copy(); minus=z94.copy(); plus[i,j]+=eps94; minus[i,j]-=eps94; numeric94[i,j]=(cross_entropy94(plus,y94).sum()-cross_entropy94(minus,y94).sum())/(2*eps94)
assert np.allclose(analytic94,numeric94,atol=1e-7)
assert np.allclose(analytic94.sum(1),0)
assert analytic94.shape==z94.shape

## 4. Padding mask 与正确 reduction

序列任务先计算每 token loss，再把 padding token 权重设 0，最终除以有效 token 数。若先对 `[batch,length]` 直接 mean，短序列会被 padding 稀释；全 padding batch 应硬失败或跳过，而不是除零返回 NaN。

ignore index 是标签合同，不应与真实类别 ID 冲突。

In [ ]:
logits_seq94=rng94.normal(size=(2,4,3)); targets_seq94=np.array([[0,1,2,-100],[2,1,-100,-100]]); valid94=targets_seq94!=-100; flat_target94=np.where(valid94,targets_seq94,0).reshape(-1); token_loss94=cross_entropy94(logits_seq94.reshape(-1,3),flat_target94).reshape(2,4)
masked_mean94=float((token_loss94*valid94).sum()/valid94.sum())
flat_logits94=torch.tensor(logits_seq94.reshape(-1,logits_seq94.shape[-1])); flat_targets_t94=torch.tensor(np.where(valid94,targets_seq94,0).reshape(-1))
torch_masked94=torch.nn.functional.cross_entropy(flat_logits94,flat_targets_t94,reduction="none").reshape(targets_seq94.shape); torch_mean94=float((torch_masked94*torch.tensor(valid94)).sum()/valid94.sum())
assert math.isclose(masked_mean94,torch_mean94,rel_tol=1e-10)
assert valid94.sum()==5 and masked_mean94>0
assert not math.isclose(masked_mean94,float((token_loss94*valid94).mean()))

## 5. Label Smoothing 与 class weight

smoothing 把 target 分布从 one-hot 改为正确类 `1-epsilon+epsilon/C`、其他类 `epsilon/C`，抑制过度自信；梯度变为 `p-q`。class weight 是按真实 target 类缩放样本 loss，二者组合时必须明确实现语义。

smoothing 不是修复错误标签的万能方法，也可能损害需要极高置信度的任务。

In [ ]:
def smooth_ce94(logits,targets,epsilon):
    z=np.asarray(logits,float); y=np.asarray(targets,int); C=z.shape[1]
    if not 0<=epsilon<1: raise ValueError("smoothing_contract")
    q=np.full_like(z,epsilon/C); q[np.arange(len(z)),y]+=1-epsilon; logp=z-logsumexp94(z)[:,None]; return -np.sum(q*logp,axis=1),softmax94(z)-q
smooth_loss94,smooth_grad94=smooth_ce94(z94,y94,.1); hard_loss94=cross_entropy94(z94,y94)
assert np.isfinite(smooth_loss94).all() and np.allclose(smooth_grad94.sum(1),0)
assert not np.allclose(smooth_loss94,hard_loss94)
assert np.allclose(smooth_ce94(z94,y94,0)[0],hard_loss94)

## 6. 自定义 PyTorch Module

Module 从 logits 计算 FP32 `logsumexp`，支持 `none/sum/mean`，再把结果转换为输入 dtype。这里不用 `F.cross_entropy`，便于观察 forward；autograd 自动从基础算子得到 `p-y`。

生产优先使用经过优化/测试的 fused kernel，但应理解它验证的数学合同。

In [ ]:
class StableCrossEntropy94(nn.Module):
    def __init__(self,reduction="mean"):
        super().__init__()
        if reduction not in {"none","sum","mean"}: raise ValueError("reduction_contract")
        self.reduction=reduction
    def forward(self,logits,targets):
        if logits.ndim!=2 or targets.shape!=(logits.shape[0],): raise ValueError("shape_contract")
        z=logits.float(); loss=torch.logsumexp(z,dim=-1)-z.gather(1,targets[:,None]).squeeze(1)
        return loss if self.reduction=="none" else (loss.sum() if self.reduction=="sum" else loss.mean())
tz94=torch.tensor(z94,dtype=torch.float64,requires_grad=True); ty94=torch.tensor(y94); module94=StableCrossEntropy94(); out94=module94(tz94,ty94); out94.backward()
assert math.isclose(float(out94),float(hard_loss94.mean()),rel_tol=1e-6)
assert np.allclose(tz94.grad.numpy(),analytic94/len(z94),atol=1e-6)
assert isinstance(module94,nn.Module)

## 7. 低精度、temperature 与常见 bug

FP16/BF16 logits 做 reduction 时通常上转 FP32，避免小概率/大词表累加误差。temperature `z/T`：T<1 更尖锐，T>1 更平滑；训练蒸馏时还要处理 `T²` 梯度缩放。

常见 bug：对 softmax 输出再次 CrossEntropy、类别维写错、mask 全部为 `-inf`、mean 分母包含 padding、label 越界。

In [ ]:
large94=torch.tensor([[80.,79.,-80.]],dtype=torch.float16); stable_fp94=StableCrossEntropy94("none")(large94,torch.tensor([0]))
assert torch.isfinite(stable_fp94).all()
entropy94=lambda p:float(-np.sum(p*np.log(np.clip(p,1e-30,1))))
p_cold94=softmax94([[2/.5,1/.5,0]] )[0]; p_hot94=softmax94([[2/2,1/2,0]])[0]
assert entropy94(p_hot94)>entropy94(p_cold94)
assert np.argmax(p_hot94)==np.argmax(p_cold94)==0

## 8. 正确性 oracle 与发布合同

loss 实现的回归集至少包含：极端 logits、平移不变、与独立实现一致、有限差分梯度、mask 分母、label 越界和低精度有限。manifest 绑定类别轴、ignore index、smoothing、weight、reduction 和 kernel/dtype。

loss 配置改变会改变梯度尺度，学习率、梯度累积和分布式平均也要一起审计。

In [ ]:
manifest94={"schema":1,"loss":"stable_softmax_cross_entropy","class_axis":-1,"reduction":"valid_token_mean","ignore_index":-100,"label_smoothing":0.,"accumulation_dtype":"float32"}; digest94=hashlib.sha256(json.dumps(manifest94,sort_keys=True,separators=(",",":")).encode()).hexdigest()
assert len(digest94)==64 and manifest94["class_axis"]==-1
assert manifest94["ignore_index"] not in range(3)
assert manifest94["reduction"]=="valid_token_mean"
print({"loss":float(out94),"grad_max_error":float(np.max(np.abs(analytic94-numeric94))),"masked_mean":masked_mean94,"sha":digest94[:12]})

## 9. 面试收束、参考与练习

回答闭环：减 max 的 softmax → LogSumExp CE → `p-y` → 数值梯度 → mask/reduction → smoothing/weight → FP32 accumulation → 极端值回归测试。能写出公式但忽略 reduction 和 dtype，工程上仍会出错。

练习：实现 soft target KL；支持 class weight；推导 Hessian；实现 sampled softmax 的偏差修正；构造全 masked 行的显式错误。

参考：[LogSumExp 技巧](https://gregorygundersen.com/blog/2020/02/09/log-sum-exp/)、[PyTorch CrossEntropyLoss 公式](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)、[Label Smoothing 经典使用](https://arxiv.org/abs/1512.00567)。